<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Notebook 99 · Final project self-check</h2>
<p><strong>SDA-DSC-211 · Tamweel Lite · Meaad Al-Marri</strong></p><p>Check your own project against the published requirements. This is a read-only self-check, not a grade or submission receipt.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>دفتر 99 · الفحص الذاتي النهائي</h2>
<p><strong>SDA-DSC-211 · تمويل لايت · ميعاد المري</strong></p><p>راجع مشروعك وفق المتطلبات المنشورة. الفحص ذاتي وللقراءة فقط؛ لا يمنح درجة ولا يرسل التسليم.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>1. Choose the complete project</h2>
<p>Use the public GitHub repository link, or one ZIP of the entire project. Do not upload the Day 5 ZIP alone. The GitHub option reads the latest default branch snapshot when you run it.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>١. اختر المشروع الكامل</h2>
<p>استخدم رابط مستودع GitHub العام، أو ZIP واحدًا يشمل المشروع بالكامل، لا حزمة اليوم الخامس وحدها. يقرأ خيار GitHub أحدث نسخة من الفرع الافتراضي وقت الفحص.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>2. Prepare the checker</h2>
<p>Run the next cell once. No packages, passwords, tokens or Drive connection are needed. The checker does not execute student code.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>٢. جهّز الفاحص</h2>
<p>شغّل الخلية التالية مرة واحدة. لا تحتاج إلى تثبيت مكتبات أو إدخال مفتاح أو ربط Drive. الفاحص لا يشغّل شيفرة الطالب.</p>
</td>
</tr>
</table>

In [ ]:
import csv
import io
import json
import math
import re
import stat
import zipfile
from pathlib import PurePosixPath

MAX_ZIP = 100 * 1024**2
MAX_TOTAL = 350 * 1024**2
MAX_TEXT = 32 * 1024**2
# The lists below reflect the current course contract, not the legacy presentation workflow.
NOTEBOOKS = ['01_baseline_boosting',
 '02_validation_tuning',
 '03_cost_sensitive_decision',
 '04_explain_calibrate',
 '05_final_model']
REQUIRED = ['README.md',
 'requirements-colab.txt',
 'constraints.txt',
 'data/data_contract.json',
 'data/data_manifest.json',
 'data/tamweel_train.csv',
 'data/tamweel_challenge.csv',
 'tamweel/__init__.py',
 'tamweel/inference.py',
 'scripts/inference.py',
 'scripts/rebuild_final.py',
 'submission/submission.csv',
 'artifacts/final_model/model.json',
 'artifacts/final_model/model_manifest.json',
 'artifacts/day5_run.json',
 'artifacts/environment.json',
 'artifacts/day5_oof_predictions.csv',
 'artifacts/day5_fold_scores.csv',
 'artifacts/ensemble_comparison.csv',
 'artifacts/final_metrics.json',
 'artifacts/final_policy.json',
 'artifacts/day5_oof_provenance.json',
 'artifacts/day5_final_provenance.json',
 'artifacts/day5_ensemble_gate.json',
 'artifacts/day5_diversity.png',
 'artifacts/day5_ensemble_comparison.png',
 'artifacts/day5_policy_regions.png',
 'artifacts/day5_calibration_fit.png',
 'artifacts/day5_challenge_capacity.png',
 'reports/MODEL_CARD.md',
 'reports/ENSEMBLE_DECISION.md',
 'notebooks/01_baseline_boosting.ipynb',
 'notebooks/02_validation_tuning.ipynb',
 'notebooks/03_cost_sensitive_decision.ipynb',
 'notebooks/04_explain_calibrate.ipynb',
 'notebooks/05_final_model.ipynb',
 'notebooks/99_final_submission_check.ipynb']
DAILY = {1: ['day1_model_comparison.csv',
     'day1_comparison_predictions.csv',
     'day1_split_membership.csv',
     'day1_learning_curves.png',
     'day1_roc_pr.png'],
 2: ['validation_summary.csv',
     'leakage_audit.csv',
     'fold_audit.csv',
     'optuna_results.csv',
     'best_params.json',
     'day2_validation_comparison.png'],
 3: ['threshold_metrics.json',
     'threshold_sweep.csv',
     'day3_oof_predictions.csv',
     'day3_region_audit.csv',
     'cost_curve.png'],
 4: ['calibration_metrics.json',
     'permutation_importance.csv',
     'day4_shap_metadata.json',
     'day4_stability_summary.json',
     'shap_beeswarm.png',
     'shap_waterfall.png',
     'reliability_curve.png']}
DAY_LABELS = {
    1: 'الأول|الاول|الأولى|الاولى', 2: 'الثاني', 3: 'الثالث',
    4: 'الرابع', 5: 'الخامس',
}
PLACEHOLDER = re.compile(
    r'^\s*(?:\[.*\]|\.{2,}|…|TODO\b.*|TBD\b.*|n/?a|(?:your answer|write your answer|اكتب(?:ي)?\s+إجابتك|ضع\s+إجابتك).*)\s*$',
    re.I,
)
PRIVATE_NAME = re.compile(
    r'(?i)^(?:\.env(?:\.(?!example$|sample$|template$).+)?|id_(?:rsa|ed25519)|'
    r'credentials\.json|challenge_labels(?:\.[^.]+)?|challenge_seeds(?:\.[^.]+)?)$'
)
SECRET_PATTERN = re.compile(
    r'(?<![A-Za-z0-9_])(?:gh[pousr]_[A-Za-z0-9]{30,}|github_pat_[A-Za-z0-9_]{50,}|'
    r'AIza[A-Za-z0-9_-]{35}|sk-(?:proj-)?[A-Za-z0-9_-]{32,})(?![A-Za-z0-9_])'
)

def review(snapshot):
    checks = []
    def add(status, item, fix):
        checks.append({'status': status, 'file': item, 'how_to_fix': fix})
    def test(item, fn):
        try:
            fn()
        except (ValueError, KeyError, TypeError, AttributeError, UnicodeError, OSError,
                RuntimeError, csv.Error, zipfile.BadZipFile, NotImplementedError) as error:
            add('FIX', item, str(error))
        else:
            add('OK', item, '')
    def require(condition, message):
        if not condition:
            raise ValueError(message)

    entries = {}
    total = 0
    for entry in snapshot.infolist():
        name = entry.orig_filename
        if '\\' in name or '\x00' in name or name.startswith('/') or re.match(r'^[A-Za-z]:', name):
            raise ValueError('Use a normal ZIP created from the project folder.')
        parts = name.rstrip('/').split('/')
        if any(part in {'', '.', '..'} for part in parts) or stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError('Remove invalid paths or symbolic links and create a new ZIP.')
        if entry.is_dir() or any(p in {'.git', '__MACOSX', '.ipynb_checkpoints', '__pycache__'} for p in parts):
            continue
        if parts[-1] == '.DS_Store':
            continue
        total += entry.file_size
        if total > MAX_TOTAL or len(entries) >= 10000:
            raise ValueError('ZIP is too large to inspect. Remove unrelated files and try again.')
        if name in entries:
            raise ValueError('ZIP contains duplicate file paths. Create a fresh ZIP.')
        entries[name] = entry
    if entries and all('/' in key for key in entries) and len({key.split('/')[0] for key in entries}) == 1:
        entries = {key.split('/', 1)[1]: value for key, value in entries.items()}
    if not entries:
        raise ValueError('The ZIP does not contain project files.')
    if 'README.md' not in entries and any(key.endswith('/README.md') for key in entries):
        add('REVIEW', 'ZIP root', 'If the results below show many missing files, ZIP the GitHub project root, not a parent folder.')
    def present(path):
        return path in entries and entries[path].file_size > 0
    def get_text(path):
        require(present(path), 'File missing or empty. Upload the original file to this path.')
        require(entries[path].file_size <= MAX_TEXT, 'File exceeds the review size limit; inspect it manually.')
        with snapshot.open(entries[path]) as stream:
            data = stream.read(MAX_TEXT + 1)
        require(len(data) <= MAX_TEXT, 'File exceeds the review size limit; inspect it manually.')
        return data.decode('utf-8-sig')
    def parse(path):
        def reject(_):
            raise ValueError('Invalid JSON: NaN and infinity are not allowed.')
        return json.loads(get_text(path), parse_constant=reject)

    for path in REQUIRED:
        if not present(path):
            add('FIX', path, 'Upload the original nonempty file from your Colab run or the extracted Day 5 bundle.')
        else:
            add('OK', path, '')
    for day, names in DAILY.items():
        for name in names:
            paths = (f'artifacts/{name}', f'evidence/day{day}/{name}')
            found = next((p for p in paths if present(p)), None)
            if found:
                add('OK', found, '')
            else:
                add('FIX', f'Day {day} / {name}', f'Upload the output to artifacts/ or evidence/day{day}/.')
    for day in range(1, 6):
        paths = (f'artifacts/day{day}_run.json', f'evidence/day{day}/day{day}_run.json')
        def run_record():
            path = next((p for p in paths if present(p)), paths[0])
            record = parse(path)
            require(record.get('technical_status') == 'TECHNICAL_READY', 'Run the lab again and export its completed FAST outputs.')
            require(not any(marker in json.dumps(record) for marker in ('EDUCATIONAL_EXAMPLE', 'EXAMPLE_ONLY', 'PRECOMPUTED')),
                    'Example or recovery files are not evidence of your own run.')
        test(f'Day {day} / run record', run_record)

    for name in NOTEBOOKS:
        path = f'notebooks/{name}.ipynb'
        def notebook():
            nb = parse(path)
            require(nb.get('nbformat') == 4, 'Save an original Colab IPYNB file.')
            cells = [c for c in nb['cells'] if c.get('cell_type') == 'code' and ''.join(c.get('source', [])).strip()]
            require(cells, 'No code cells found.')
            require(not any(o.get('output_type') == 'error' for c in cells for o in c.get('outputs', [])),
                    'Fix saved red errors, run again, and save the executed notebook.')
            require(any(c.get('outputs') for c in cells), 'Run the notebook and save the outputs before uploading.')
            require(all(type(c.get('execution_count')) is int and c['execution_count'] > 0 for c in cells),
                    'Some code cells were not run. Use Runtime → Run all and save the results.')
        test(path + ' (executed)', notebook)

    def answers():
        text = re.sub(r'(?m)<!--.*?-->|^\s*\x60{3}[^\n]*\n.*?^\s*\x60{3}[^\n]*$', '', get_text('README.md'), flags=re.S)
        lines = text.splitlines()
        heading = next((i for i, line in enumerate(lines) if re.match(
            r'^\s*#{1,6}\s+.*(?:daily\s+answers|الإجابات\s+اليومية|الاجابات\s+اليومية)', line, re.I)), None)
        require(heading is not None, 'Add the section Daily answers | الإجابات اليومية and answer Days 1–5.')
        level = len(lines[heading].lstrip().split()[0])
        found = {}
        active = None
        for line in lines[heading + 1:]:
            clean = re.sub(r'^[#*>|_\-\s]+', '', line).replace('**', '').replace('__', '')
            for day, label in DAY_LABELS.items():
                match = re.match(rf'(?i)(?:day[\s_-]*0?{day}|اليوم\s*(?:{label}|{day}|{chr(0x660+day)})|{day}[.)])(?=$|[\s:：|–—).\-])', clean)
                if match:
                    active = day
                    rest = clean[match.end():].strip(' :：|–—-)')
                    rest = re.sub(rf'^\|?\s*(?:اليوم\s+(?:{label})|Day\s+0?{day})[\s:：|–—-]*', '', rest, flags=re.I)
                    found.setdefault(day, []).append(rest)
                    break
            else:
                marker = re.match(r'^\s*(#{1,6})\s+', line)
                if marker:
                    active = None
                    if len(marker[1]) <= level:
                        break
                elif active and clean and not re.fullmatch(r'[\s|:\-]+', clean):
                    found[active].append(clean.strip(' |'))
        missing = [str(day) for day in DAY_LABELS
                   if not any(part and not PLACEHOLDER.fullmatch(part) and part not in {'-', '—', '…'}
                              for part in found.get(day, []))]
        if missing:
            add('REVIEW', 'README.md / Days ' + ', '.join(missing),
                'Check these answers manually if your format is different. Label each entry Day N or اليوم N and write your own answer.')
    test('README.md / daily answers', answers)

    def predictions():
        expected = {'application_id', 'probability', 'decision'}
        reader = csv.DictReader(io.StringIO(get_text('submission/submission.csv'), newline=''), strict=True)
        require(len(reader.fieldnames or []) == 3 and set(reader.fieldnames or []) == expected,
                'Use the generated CSV with application_id, probability and decision.')
        rows = list(reader)
        require(len(rows) == 2500, 'Expected 2,500 predictions. Export the full challenge batch from Day 5.')
        ids, flags = set(), 0
        for row in rows:
            require(set(row) == expected and all(v is not None for v in row.values()), 'A CSV row has the wrong number of fields.')
            key = row['application_id'].strip()
            require(key and key not in ids, 'A request ID is missing or duplicated. Re-export the file; do not edit it manually.')
            ids.add(key)
            try:
                value = float(row['probability'])
            except ValueError:
                raise ValueError('A probability is not numeric. Re-export the file.') from None
            require(math.isfinite(value) and 0 <= value <= 1, 'A probability is outside the range 0 to 1.')
            require(row['decision'].strip() in {'0', '1'}, 'Use the generated binary decision: 0 or 1.')
            flags += row['decision'].strip() == '1'
        require(flags <= 300, 'Review flags must not exceed 12% of the 2,500 requests (300).')
        challenge = csv.DictReader(io.StringIO(get_text('data/tamweel_challenge.csv'), newline=''))
        expected_ids = {str(row.get('application_id') or '').strip() for row in challenge}
        require(ids == expected_ids, 'Request IDs do not match the full official challenge batch.')
    test('submission/submission.csv (predictions)', predictions)

    for path in ('artifacts/final_policy.json', 'artifacts/final_metrics.json', 'artifacts/final_model/model.json',
                 'artifacts/final_model/model_manifest.json'):
        def final_json():
            obj = parse(path)
            require(isinstance(obj, dict) and obj, 'Re-export a valid, nonempty JSON file from Day 5.')
            if path.endswith('final_policy.json'):
                require(obj.get('cost_policy') == {'false_negative_cost': 10.0, 'false_positive_cost': 1.0, 'max_flag_fraction': .12},
                        'Rebuild the final policy with the course cost and review-cap settings.')
        test(path + ' (content)', final_json)

    skipped = False
    for path, entry in entries.items():
        if PRIVATE_NAME.fullmatch(PurePosixPath(path).name):
            add('FIX', path, 'Remove the private file before submission.')
            continue
        if PurePosixPath(path).suffix.lower() not in {'.md', '.ipynb', '.py', '.json', '.csv', '.txt', '.yml', '.yaml'}:
            continue
        if entry.file_size > MAX_TEXT:
            skipped = True
            continue
        try:
            if entry.file_size and SECRET_PATTERN.search(get_text(path)):
                add('REVIEW', path, 'Possible access key. Inspect privately; if real, revoke it and remove it. The value is not printed.')
        except (ValueError, OSError, RuntimeError, zipfile.BadZipFile, NotImplementedError):
            skipped = True
    if skipped:
        add('REVIEW', 'Security spot check', 'Some files could not be scanned. Inspect them manually for credentials and personal data.')
    return checks

def summarize(checks):
    fixes = sum(row['status'] == 'FIX' for row in checks)
    reviews = sum(row['status'] == 'REVIEW' for row in checks)
    status = 'NEEDS_WORK' if fixes else 'REVIEW_NOTES' if reviews else 'SELF_CHECK_COMPLETE'
    return {'status': status, 'files_to_fix': fixes, 'manual_review_notes': reviews, 'checks': checks}


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>3. Inspect the snapshot</h2>
<p>Paste the repository link in <code>PROJECT_URL</code>, or leave both fields blank and upload a complete ZIP. The checker reads notebooks 01–05, the five README answers, key daily and Day 5 outputs, predictions and obvious credential warnings.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>٣. افحص نسخة المشروع</h2>
<p>ضع الرابط في <code>PROJECT_URL</code>، أو اترك الحقلين فارغين وارفع ZIP كاملًا. يراجع الفاحص الدفاتر 01–05، وإجابات README الخمس، والمخرجات اليومية والنهائية، والتنبؤات، والمؤشرات الظاهرة على وجود مفاتيح وصول.</p>
</td>
</tr>
</table>

In [ ]:
# Paste your public GitHub repository link, or enter a path to a ZIP already in Colab Files.
PROJECT_URL = ""
PROJECT_ZIP = ""

import io
import re
import urllib.request
from urllib.error import HTTPError, URLError
from urllib.parse import urlsplit
from pathlib import Path

def load_snapshot(url, local_path):
    url, local_path = url.strip(), local_path.strip()
    if url and local_path:
        raise ValueError('Choose one input: a public GitHub link or one project ZIP.')
    if not url and not local_path:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload exactly one complete project ZIP.')
        name, payload = next(iter(uploaded.items()))
        if not name.lower().endswith('.zip') or len(payload) > MAX_ZIP:
            raise ValueError('Choose one ZIP below 100 MiB.')
        return io.BytesIO(payload), 'Uploaded project ZIP'
    if local_path:
        path = Path(local_path)
        if path.suffix.lower() != '.zip' or not path.is_file():
            raise ValueError('Choose an existing .zip file in Colab Files.')
        if path.stat().st_size > MAX_ZIP:
            raise ValueError('Choose one ZIP below 100 MiB.')
        return path, 'Local project ZIP'
    if re.fullmatch(r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+', url):
        url = 'https://github.com/' + url
    parsed = urlsplit(url)
    match = re.fullmatch(r'/([A-Za-z0-9_.-]+)/([A-Za-z0-9_.-]+)/?', parsed.path)
    if parsed.scheme != 'https' or parsed.netloc.lower() not in {'github.com', 'www.github.com'} or not match:
        raise ValueError('Paste the GitHub repository link only: https://github.com/owner/repository')
    owner, repo = match.groups()
    repo = repo.removesuffix('.git')
    if any(part in {'', '.', '..'} for part in (owner, repo)):
        raise ValueError('Invalid GitHub account or repository name.')
    try:
        with urllib.request.urlopen(f'https://codeload.github.com/{owner}/{repo}/zip/HEAD', timeout=30) as response:
            payload = response.read(MAX_ZIP + 1)
    except (HTTPError, URLError, TimeoutError) as error:
        raise ValueError('Could not open the public GitHub snapshot. Check the link and visibility, or upload the complete ZIP.') from error
    if len(payload) > MAX_ZIP:
        raise ValueError('ZIP exceeds 100 MiB; remove unrelated large files and use a complete project ZIP.')
    return io.BytesIO(payload), f'{owner}/{repo} (default branch)'

try:
    source, label = load_snapshot(PROJECT_URL, PROJECT_ZIP)
    with zipfile.ZipFile(source) as snapshot:
        report = summarize(review(snapshot))
    report['source'] = label
except (ValueError, OSError, RuntimeError, zipfile.BadZipFile, NotImplementedError) as error:
    print('INPUT_ERROR | تعذر فتح المشروع:', error)
else:
    print(label)
    print(report['status'], '| files to fix:', report['files_to_fix'], '| manual notes:', report['manual_review_notes'])
    for row in report['checks']:
        if row['status'] != 'OK':
            print(f"{row['status']} | {row['file']} | {row['how_to_fix']}")
    if report['status'] == 'SELF_CHECK_COMPLETE':
        print('No missing items were found by the checks performed. This is not a grade or a submission receipt.')
    report_path = Path('tamweel_lite_self_check.json')
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Download the report from Colab Files:', report_path)


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>4. Read the findings</h2>
<p><strong>FIX:</strong> correct the specified item. <strong>REVIEW:</strong> inspect it manually. The checker cannot judge the quality of an answer, verify attendance or certify model performance.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>٤. اقرأ الملاحظات</h2>
<p><strong>FIX:</strong> صحح البند المحدد. <strong>REVIEW:</strong> راجعه يدويًا. الفاحص لا يحكم على جودة الإجابة، ولا يتحقق من الحضور، ولا يثبت أداء النموذج.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>5. Save and submit separately</h2>
<p>Download the JSON report from Colab Files if needed. After any project change, rerun this self-check. The <a href="https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/SUBMISSION_GUIDE.md">submission guide</a> explains the remaining steps. No presentation or submission tag is required.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>٥. احفظ التقرير وسلّم بشكل مستقل</h2>
<p>نزّل تقرير JSON من لوحة Files عند الحاجة، وأعد الفحص بعد أي تعديل. اتبع <a href="https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/SUBMISSION_GUIDE.md">دليل التسليم</a> لإكمال الخطوات وإرسال رابطك عبر القناة الخاصة. لا يلزم عرض تقديمي أو وسم تسليم.</p>
</td>
</tr>
</table>